# Kafka

In [13]:
!pip cache purge --quiet

In [14]:
import pandas as pd
import plotly.graph_objs as go
import time

from IPython.display import display

<div class="alert alert-block alert-warning">
    <b class="fa fa-solid fa-exclamation-circle"></b>
    <div>
        <p><b>Action Required</b></p>
        <p>Select the database from the drop-down menu at the top of this notebook. It updates the <b>connection_url</b> which is used by SQLAlchemy to make connections to the selected database.</p>
    </div>
</div>

In [15]:
from sqlalchemy import *

db_connection = create_engine(connection_url)

In [16]:
refresh_interval = 5
num_records = 1000

In [17]:
def get_latest_sensor_data():
    """
    Query latest temperature for each sensor and return as a DataFrame
    with timestamp converted to pandas datetime.
    """
    df = pd.read_sql("""
        SELECT t.id, t.temperature, t.timestamp, s.latitude, s.longitude
        FROM sensors s
        JOIN temperatures t
            ON t.id = s.id
            AND t.timestamp = (
                SELECT MAX(timestamp)
                FROM temperatures
                WHERE id = s.id
            );
    """, db_connection)

    df["timestamp"] = pd.to_datetime(df["timestamp"], unit = "ms")
    return df

In [18]:
# Initial data
df = get_latest_sensor_data()

# Initial figure
fig = go.FigureWidget(
    go.Scattergeo(
        lat = df["latitude"],
        lon = df["longitude"],
        mode = "markers",
        marker = dict(
            size = 8,
            color = df["temperature"],
            colorscale = "Turbo",
            cmin = df["temperature"].min(),
            cmax = df["temperature"].max(),
            colorbar = dict(title = "Temperature")
        ),
        text = df["id"]
    )
)

fig.update_layout(
    title = "Live Sensor Temperature Readings",
    title_x = 0.5,
    geo = dict(
        showland = True,
        landcolor = "LightGreen",
        showcountries = True,
        projection_type = "natural earth"
    ),
    width = 800,
    height = 600
)

display(fig)

FigureWidget({
    'data': [{'lat': array([ 32.56644742, -63.12474396, -61.77429585, ...,  17.61166995,
                             50.58889025,  52.08270089]),
              'lon': array([  64.61752437,   -8.24556083, -104.3740063 , ..., -164.5880101 ,
                             110.87047553,  -87.40603318]),
              'marker': {'cmax': 24.95,
                         'cmin': -13.76,
                         'color': array([19.88,  4.79,  7.6 , ..., 22.56,  7.78,  7.98]),
                         'colorbar': {'title': {'text': 'Temperature'}},
                         'colorscale': [[0.0, '#30123b'], [0.07142857142857142,
                                        '#4145ab'], [0.14285714285714285,
                                        '#4675ed'], [0.21428571428571427,
                                        '#39a2fc'], [0.2857142857142857,
                                        '#1bcfd4'], [0.35714285714285715,
                                        '#24eca6'], [0.42857142857

In [19]:
# Loop to update the data
try:
    while True:
        df = get_latest_sensor_data()

        # Update the figure data in place
        fig.data[0].text = df["id"]
        fig.data[0].lat = df["latitude"]
        fig.data[0].lon = df["longitude"]
        fig.data[0].marker.color = df["temperature"]
        
        time.sleep(refresh_interval)

except KeyboardInterrupt:
    print("Stopped by user.")

Stopped by user.
